# Training an anomaly detector

This notebook trains an autoencoder on zero-bias events and saves it to `output/<RUN_NAME>/`, ready for
[`evaluate.ipynb`](evaluate.ipynb). Run it top to bottom as it is to train the baseline model, `dense-ae`.
Then make it yours: change the inputs, the model and the training settings, and fill in the TODOs.

<!-- TODO(organisers): explain the autoencoder idea: learn to reconstruct normal events, and flag events that reconstruct badly. -->

In [ ]:
import math

import numpy as np
import torch
import yaml
from torch import nn

import dataset
import plotting
from dataloader import create_datasets
from models.registry import REGISTRY, register
from utils import fit, get_device, save_model, summary

## 1. Choose the inputs

Each event is turned into a fixed-size array: for every collection, the highest-Et objects are kept, up to the
number of slots you give it in `LAYOUT`, and empty slots are padded with zeros. Each slot holds the `FEATURES` you choose.

- Collections: `muons`, `jets`, `egammas`, `taus` (up to 8 muons and 12 of the others per event) and the energy sums
  `MET`, `MHT`, `FET`, `FHT` (Et, phi) and `ET`, `HT` (Et only), one per event.
- Features: `Et`, `eta`, `phi`. A collection without a feature (e.g. eta for MET) gets a constant there.

The model sees `sum(LAYOUT.values()) * len(FEATURES)` numbers per event. Every input costs parameters in the first and
last layer, and resources on the FPGA. Think about it: which objects matter for the signals you want to find?
Do the most energetic jets carry most of the information? Does phi tell you anything on its own?

In [ ]:
DATA_CONFIG = "config/baseline.yml"  # which samples to train and test on, see config/

# Number of objects kept per collection (highest Et first)
LAYOUT = {"MET": 1, "muons": 4, "jets": 10, "egammas": 12, "taus": 12}
# Features of each object
FEATURES = ["Et", "eta", "phi"]

n_inputs = sum(LAYOUT.values()) * len(FEATURES)
print(f"{sum(LAYOUT.values())} slots x {len(FEATURES)} features = {n_inputs} inputs per event")

## 2. Choose the model

The baseline, `dense-ae`, lives in [`models/autoencoder.py`](models/autoencoder.py). Write your own model in the cell below
(or in `models/autoencoder.py`, to share it with your team), then set `model_type` to its name.

An autoencoder must output one number for every input: its reconstruction. Remember that the model has to fit on an FPGA,
so watch the number of parameters (and see [`convert_hls4ml.ipynb`](convert_hls4ml.ipynb)).

**The challenge:** find a model that is as cheap as possible on the FPGA without losing too much performance.
`model_cost` estimates the cost without building the firmware: the number of parameters, the *bit operations*
(BOPs, roughly the FPGA area) and the number of dense layers (roughly the latency). See `evaluate.ipynb` for details.
Ways to lower the cost: fewer inputs, fewer or narrower layers, and fewer bits. `dense-qae` is `dense-ae` trained with
6-bit weights and activations ([Brevitas](https://github.com/Xilinx/brevitas) *quantization-aware training*): compare their BOPs.

In [ ]:
@register("my-model", input_type="flat")
def my_model(input_shape):
    """Your autoencoder. input_shape is the shape of one event, e.g. (117,)."""
    n_in = math.prod(input_shape)
    # TODO: return an nn.Sequential that maps n_in inputs to n_in outputs.
    #       Look at dense_autoencoder in models/autoencoder.py for an example. Ideas: fewer or narrower
    #       layers, a smaller latent space, other activation functions, batch normalization,
#       quantized layers from brevitas.nn (see dense-qae), ...
    raise NotImplementedError("Write my_model, or set model_type = 'dense-ae' below")

In [ ]:
model_type = "dense-ae"  # "my-model" once you have written it

print("Available models:", list(REGISTRY))
model_cfg = REGISTRY[model_type]

## 3. Load the data

In [ ]:
train, val, _ = create_datasets(DATA_CONFIG, layout=LAYOUT, features=FEATURES, input_type=model_cfg["input_type"])
print(f"{len(train):,} training events, {len(val):,} validation events, input shape {train.x.shape[1:]}")

In [ ]:
# The model inputs, after the scaling defined in the config
plotting.plot_data_distributions(train, feature_names=FEATURES);

In [ ]:
model = model_cfg["build"](train.x.shape[1:])

summary(model, train.x.shape[1:])

## 4. Train

The loss is the reconstruction error of each event, ignoring padded objects. `fit` (in [`utils.py`](utils.py), have a look at
the training loop) minimises it with the Adam optimiser, and evaluates it on the validation events after every epoch.
Training stops after `N_EPOCHS`; to stop when the validation loss no longer improves (and keep the best weights),
set `PATIENCE`: the number of epochs without improvement after which training stops (*early stopping*).

In [ ]:
device = get_device()  # the GPU if there is one
print("Training on", device)

In [ ]:
N_EPOCHS = 10
PATIENCE = None  # TODO: use early stopping (e.g. PATIENCE = 3) and train for more epochs

history = fit(model, train, val, n_epochs=N_EPOCHS, batch_size=1024, learning_rate=0.001,
              patience=PATIENCE, device=device)
plotting.plot_loss_curve(history);

## 5. Save

The model is saved (its weights, and the function that builds it) together with `inputs.yml`, the inputs you chose
above, so that the evaluation and hls4ml notebooks build the same inputs for it.

In [ ]:
from pathlib import Path

# Give every variant you train its own name (e.g. "dense-ae-no-taus"): it is the folder the model
# is saved in, and the name it appears under in evaluate.ipynb.
RUN_NAME = model_type

out_dir = Path("output", RUN_NAME)
out_dir.mkdir(parents=True, exist_ok=True)
save_model(model, model_cfg["build"], train.x.shape[1:], out_dir / "best_model.pt")
inputs = {"layout": LAYOUT, "features": FEATURES, "input_type": model_cfg["input_type"], "data_config": DATA_CONFIG}
with open(out_dir / "inputs.yml", "w") as f:
    yaml.safe_dump(inputs, f, sort_keys=False)
print(f"Saved to {out_dir}/")

## Next steps

<!-- TODO(organisers): what to try next. -->

- Evaluate your model in [`evaluate.ipynb`](evaluate.ipynb) and save it as a trial.
- Try other inputs and models, and compare them all in the trials plot at the end of `evaluate.ipynb`.